In [1]:
import argparse
import logging
from espnet2.bin.asr_inference import Speech2Text
import gc
import os
import csv
from speechbrain.inference.ASR import WhisperASR
from speechbrain.inference.ASR import EncoderASR
from utils.read_transcription import *
from utils.normalise_text import *
from pathlib import Path
from hyperpyyaml import load_hyperpyyaml
import librosa
import torch
from utils.meta import get_audio_info
from utils.apply_vad import *
from utils.list_files import list_files
from utils.VAD_chunk import *
from utils.wer_chunk import wer_chunk
from utils.logging_config import setup_logging
from utils.wer_segment import wer_segment
models = ["wav2vec","whisper-VAD-chunk","whisper-large","whisper-medium","whisper-large-VAD-chunk","wav2vec2-VAD-chunk"]
import gc
gc.collect()

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


0

In [2]:
wer_hparams = load_hyperpyyaml("""wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats""")


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
w2v = EncoderASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", run_opts={"device":"cuda:0"})
whisper_med = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr", run_opts={"device":"cuda:0"})
whisper_large = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr", run_opts={"device":"cuda:0"})

speech2text_ester = Speech2Text(
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer_config.yaml",
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer.pth",
            device="cuda:0"
        )
speech2text = Speech2Text(
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR_config.yaml",
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR.pth",
            device="cuda:0"
        )

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - Wav2Vec2Model is frozen.


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


In [8]:
wav_data="/vol/corpora/Daoudi/Data/Reading/MSA" 
ref_trans= "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcript_Monologue_final/PD"
txt = "/vol/corpora/Daoudi/Data/text_chevre.txt"
csv_path = "khalid/khalid_MSA_read.csv"
pred_folder = "khalid"

In [11]:
os.listdir(wav_data)

['1MSA2-HQZQ-chevre.wav',
 '2MSA-BDID-chevre.wav',
 '1MSA2-MIMH-chevre.wav',
 '._2MSA-TDHV-chevre.wav',
 '2MSA2-IZHO-chevre.wav',
 '2MSA2-BQME-chevre.wav',
 '1MSA-HGJK-chevre.wav',
 '1MSA-ESWQ-chevre.wav.old',
 '1MSA-JAJC-chevre.wav',
 '2MSA2-VCSC-chevre.wav.old',
 '1MSA2-KHDK-chevre.wav',
 '2MSA2-WCGQ-chevre.wav',
 '1MSA-TJHD-chevre.wav',
 '2MSA2-VCSC-chevre_bis.wav',
 '1MSA-EDKI-chevre.wav',
 '1MSA-MCKC-chevre_bis.wav',
 '1MSA-FXAY-chevre.wav',
 '2MSA-DWSN-chevre.wav',
 '1MSA2-ADOD-chevre.wav.old',
 '2MSA-PBLQ-chevre.wav',
 '2MSA2-GCKC-chevre.wav.old',
 '2MSA-TDHV-chevre.wav',
 '1MSA-PLZK-chevre.wav',
 '1MSA-ESWQ-chevre_bis.wav',
 '2MSA-ZQMQ-chevre.wav',
 '2MSA2-VUWZ-chevre.wav',
 '2MSA-FTCM_chevre.wav',
 '1MSA2-PDTD-chevre.wav',
 '1MSA2-ADOD-chevre_bis.wav',
 '._2MSA-PBLQ-chevre.wav',
 '2MSA-LGIS_chevre.wav',
 '2MSA2-GCKC-chevre_bis.wav',
 '1MSA-YWWF-chevre.wav',
 '1MSA-MCKC-chevre.wav.old']

In [10]:
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):
            if f.endswith(".txt") and not w.endswith("old"):
                if f.split("-")[1].split(".")[0] in w:
                    tg_to_wav[f] = w
           

    return tg_to_wav
tg_to_wav = list_files(ref_trans)
tg_to_wav


{}

In [28]:
len(tg_to_wav)

32

In [12]:
import re

def clean_transcription(text):
    # 1. Garder seulement la partie après ***
    if "***" in text:
        text = text.split("***", 1)[1]

    # 2. Supprimer contenu entre [] et ()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 3. Supprimer # mais garder contenu
    text = text.replace("#", "")

    # 4. Nettoyage des espaces multiples
    text = re.sub(r"\s+", " ", text)

    # 5. Nettoyage des espaces en début/fin de lignes
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    return (" ".join(lines)).split(" ")




In [13]:
with open(txt,"r") as f:
    t=f.read()

In [15]:
import os
import csv
import torch
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.metric_stats import ErrorRateStats
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# ===================== WER CONFIG =====================
wer_hparams = load_hyperpyyaml("""
wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats
""")

# ===================== WER FUNCTION =====================
def wer_chunk(results, words):
    hyp = ""
    for r in results:
        hyp += r["text"] + " "
    ref = " ".join(words)

    hyp_norm = normalization(hyp)
    ref_norm = normalization(ref)

    wer_hparams["wer_stats"].clear()
    wer_hparams["wer_stats"].append(
        ids=[0],
        predict=[hyp_norm],
        target=[ref_norm]
    )

    stats = wer_hparams["wer_stats"].summarize()

    S = stats["substitutions"]
    D = stats["deletions"]
    I = stats["insertions"]
    WER = stats["WER"]

    print(f'WER={WER:.4f}, S={S}, D={D}, I={I}')

    return ref_norm, hyp_norm, WER, S, D, I


# ===================== CORPUS STATS =====================
corpus_stats = {
    "w2vec": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper_large": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_cv": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_ester": {"S": 0, "D": 0, "I": 0, "N": 0},
    "hmm": {"S": 0, "D": 0, "I": 0, "N": 0},
}

# ===================== CSV =====================
with open(csv_path, "w", newline="", encoding="utf-8") as f:

    fieldnames = [
        "filename", "duration_sec", "samplerate", "channels",

        "trans_w2vec_vad_chunk", "WER_w2vec_vad_chunk", "S_w2vec", "D_w2vec", "I_w2vec",
        "trans_whisper_vad_chunk", "WER_whisper_vad_chunk", "S_whisper", "D_whisper", "I_whisper",
        "trans_whisper_large_vad_chunk", "WER_whisper_large_vad_chunk", "S_whisper_large", "D_whisper_large", "I_whisper_large",
        "trans_conf_cv_vad_chunk", "WER_conf_cv_vad_chunk", "S_conf_cv", "D_conf_cv", "I_conf_cv",
        "trans_conf_ester_vad_chunk", "WER_conf_ester_vad_chunk", "S_conf_ester", "D_conf_ester", "I_conf_ester",
        "trans_hmm_tdnn_vad_chunk", "WER_hmm_tdnn_vad_chunk", "S_hmm", "D_hmm", "I_hmm"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    # ===================== LOOP =====================
    #for tg, wav in tg_to_wav.items():
    for wav in sorted(os.listdir(wav_data)):
        if not wav.endswith(".old") and not wav.startswith("."):
            wav_file = os.path.join(wav_data, wav)
            #trans_file = os.path.join(ref_trans, tg)
    
            #if not (os.path.exists(wav_file) and os.path.exists(trans_file)):
                #continue
    
            info = get_audio_info(wav_file)
            row = {"filename": wav, **info}
    
            # Load audio
            audio_np, sr = read_audio_16k(wav_file)
            wav_tensor = torch.from_numpy(audio_np)
    
            # VAD
            chunks = vad_chunk_with_timestamps(wav_tensor)
    
            # Reference
            #with open(trans_file, "r", encoding="utf-8") as tf:
                #text = tf.read()
            words = clean_transcription(t)
    
            ref_len = len(normalization(" ".join(words)))
    
            # ======== W2VEC ========
            results = whisper_transcribe_chunks(w2v, "wav2vec2-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_w2vec_vad_chunk"] = pred
            row["WER_w2vec_vad_chunk"] = wer
            row["S_w2vec"], row["D_w2vec"], row["I_w2vec"] = S, D, I
    
            corpus_stats["w2vec"]["S"] += S
            corpus_stats["w2vec"]["D"] += D
            corpus_stats["w2vec"]["I"] += I
            corpus_stats["w2vec"]["N"] += ref_len
    
            # ======== WHISPER MED ========
            results = whisper_transcribe_chunks(whisper_med, "whisper-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_whisper_vad_chunk"] = pred
            row["WER_whisper_vad_chunk"] = wer
            row["S_whisper"], row["D_whisper"], row["I_whisper"] = S, D, I
    
            corpus_stats["whisper"]["S"] += S
            corpus_stats["whisper"]["D"] += D
            corpus_stats["whisper"]["I"] += I
            corpus_stats["whisper"]["N"] += ref_len
    
            # ======== WHISPER LARGE ========
            results = whisper_transcribe_chunks(whisper_large, "whisper-large-VAD-chunk", wav_tensor, chunks)
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_whisper_large_vad_chunk"] = pred
            row["WER_whisper_large_vad_chunk"] = wer
            row["S_whisper_large"], row["D_whisper_large"], row["I_whisper_large"] = S, D, I
    
            corpus_stats["whisper_large"]["S"] += S
            corpus_stats["whisper_large"]["D"] += D
            corpus_stats["whisper_large"]["I"] += I
            corpus_stats["whisper_large"]["N"] += ref_len
    
            # ======== CONFORMER CV ========
            
            results = espnet_transcribe_chunks(speech2text, wav_tensor, chunks, sr=16000)
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_conf_cv_vad_chunk"] = pred
            row["WER_conf_cv_vad_chunk"] = wer
            row["S_conf_cv"], row["D_conf_cv"], row["I_conf_cv"] = S, D, I
    
            corpus_stats["conf_cv"]["S"] += S
            corpus_stats["conf_cv"]["D"] += D
            corpus_stats["conf_cv"]["I"] += I
            corpus_stats["conf_cv"]["N"] += ref_len
    
            # ======== CONFORMER ESTER ========
            
            results = espnet_transcribe_chunks(speech2text_ester, wav_tensor, chunks, sr=16000)
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_conf_ester_vad_chunk"] = pred
            row["WER_conf_ester_vad_chunk"] = wer
            row["S_conf_ester"], row["D_conf_ester"], row["I_conf_ester"] = S, D, I
    
            corpus_stats["conf_ester"]["S"] += S
            corpus_stats["conf_ester"]["D"] += D
            corpus_stats["conf_ester"]["I"] += I
            corpus_stats["conf_ester"]["N"] += ref_len
    
            # ======== HMM-TDNN ========
            results = hmmtdnn_transcribe_chunks(
                "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/asr_FR_kaldi_hmm_tdnn.sh",
                wav_tensor, chunks,
                "/vol/experiments3/imbenamor/TAPAS-FRAIS/logs/transcription/rhap",
                sr=16000
            )
    
            _, pred, wer, S, D, I = wer_chunk(results, words)
    
            row["trans_hmm_tdnn_vad_chunk"] = pred
            row["WER_hmm_tdnn_vad_chunk"] = wer
            row["S_hmm"], row["D_hmm"], row["I_hmm"] = S, D, I
    
            corpus_stats["hmm"]["S"] += S
            corpus_stats["hmm"]["D"] += D
            corpus_stats["hmm"]["I"] += I
            corpus_stats["hmm"]["N"] += ref_len
    
            writer.writerow(row)


# ===================== FINAL CORPUS RESULTS =====================
print("\n===== FINAL CORPUS WER =====")

for model, stats in corpus_stats.items():
    S, D, I, N = stats["S"], stats["D"], stats["I"], stats["N"]

    wer = (S + D + I) / N if N > 0 else 0

    print(f"{model}: WER={wer:.4f} | S={S}, D={D}, I={I}, N={N}")

WER=49.2958, S=23, D=12, I=0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


WER=69.0141, S=25, D=23, I=1
WER=71.8310, S=13, D=37, I=1
WER=60.5634, S=33, D=5, I=5
WER=77.4648, S=39, D=1, I=15
WER=61.9718, S=34, D=6, I=4
WER=21.1268, S=12, D=3, I=0
WER=28.1690, S=18, D=0, I=2
WER=16.9014, S=12, D=0, I=0
WER=18.3099, S=13, D=0, I=0
WER=14.0845, S=9, D=0, I=1
WER=26.7606, S=15, D=4, I=0
WER=26.7606, S=15, D=2, I=2
WER=29.5775, S=15, D=3, I=3
WER=21.1268, S=11, D=1, I=3
WER=19.7183, S=12, D=1, I=1
WER=38.0282, S=23, D=2, I=2
WER=46.4789, S=27, D=6, I=0
WER=46.4789, S=24, D=9, I=0
WER=42.2535, S=22, D=7, I=1
WER=29.5775, S=17, D=4, I=0
WER=38.0282, S=23, D=4, I=0
WER=50.7042, S=29, D=5, I=2
WER=67.6056, S=31, D=17, I=0
WER=25.3521, S=13, D=5, I=0
WER=59.1549, S=16, D=25, I=1
WER=43.6620, S=14, D=14, I=3
WER=43.6620, S=28, D=1, I=2
WER=25.3521, S=15, D=1, I=2
WER=43.6620, S=22, D=8, I=1
WER=18.3099, S=11, D=2, I=0
WER=56.3380, S=7, D=32, I=1
WER=22.5352, S=14, D=2, I=0
WER=36.6197, S=21, D=1, I=4
WER=35.2113, S=22, D=2, I=1
WER=32.3944, S=20, D=2, I=1
WER=12.6761, S=